# DL Competition 01 — Image Classification (PyTorch)
**Intel Scene Classification — 6 Classes**

**Approach: Multi-Layer Perceptron (MLP)** — Fully connected neural network without convolutional layers.

## Step 0: Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os
import pandas as pd
from tqdm import tqdm

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## Step 1: Data Loading and Preprocessing

### Preprocessing Pipeline
- Grayscale (1 channel)
- Resize to 150×150
- **Data Augmentation (training):** RandomHorizontalFlip, RandomVerticalFlip, RandomRotation, RandomAffine, ColorJitter (brightness on grayscale)
- Normalize with mean=0.5, std=0.5

Since MLPs lack the spatial inductive bias of CNNs, strong data augmentation is critical to improve generalization.

In [ ]:
# Training transform WITH augmentation (more aggressive for MLP)
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((150, 150)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Validation/Test/CompTest transform WITHOUT augmentation
eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

### Load Training Dataset

In [ ]:
train_path = '/kaggle/input/intel-image-classification/seg_train/seg_train'
train_dataset = datasets.ImageFolder(root=train_path, transform=train_transform)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

print(f"Training samples: {len(train_dataset)}")
print(f"Classes: {train_dataset.class_to_idx}")

# Visualize samples
for images, labels in train_loader:
    images_vis = images * 0.5 + 0.5
    plt.figure(figsize=(10, 10))
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images_vis[i][0], cmap='gray')
        plt.title(f"Label: {labels[i].item()}")
        plt.axis('off')
    plt.suptitle("Training Samples (with Augmentation)")
    plt.show()
    break

### Load Validation and Test Datasets (Stratified Split)

In [ ]:
val_path = '/kaggle/input/intel-image-classification/seg_test/seg_test'
val_dataset_full = datasets.ImageFolder(root=val_path, transform=eval_transform)

targets = np.array(val_dataset_full.targets)
sample_size = 100
num_iterations = 2

splitter = StratifiedShuffleSplit(n_splits=num_iterations, test_size=sample_size, random_state=42)

sample_datasets = []
for train_idx, sample_idx in splitter.split(np.zeros(len(targets)), targets):
    sample_datasets.append(Subset(val_dataset_full, sample_idx))

val_dataset = sample_datasets[0]
test_dataset = sample_datasets[1]

val_loader = DataLoader(val_dataset, batch_size=100, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False)

print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

### Load Competition Test Dataset (Unlabeled)

In [ ]:
image_directory = '/kaggle/input/intel-image-classification/seg_pred/seg_pred'
image_files = os.listdir(image_directory)
image_files = [f for f in image_files if f.endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
image_files = image_files[:200]

class SelectedFilesDataset(Dataset):
    def __init__(self, root_dir, file_names, transform=None):
        self.root_dir = root_dir
        self.file_names = file_names
        self.transform = transform

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        file_path = os.path.join(self.root_dir, self.file_names[idx])
        image = Image.open(file_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.file_names[idx]

comp_test_dataset = SelectedFilesDataset(image_directory, image_files, transform=eval_transform)
comp_test_loader = DataLoader(comp_test_dataset, batch_size=100, shuffle=False)

print(f"Competition test samples: {len(comp_test_dataset)}")

## Step 2: Model Definition — MLP (Multi-Layer Perceptron)

A fully connected neural network that flattens the input image and processes it through dense layers.

**Architecture:**
- Input: Flattened 1×150×150 = 22,500 features
- Hidden Layer 1: 22,500 → 1024 + BatchNorm + ReLU + Dropout(0.4)
- Hidden Layer 2: 1024 → 512 + BatchNorm + ReLU + Dropout(0.4)
- Hidden Layer 3: 512 → 256 + BatchNorm + ReLU + Dropout(0.3)
- Output: 256 → 6 classes

In [ ]:
class SceneMLP(nn.Module):
    def __init__(self, input_size=1*150*150, n_classes=6):
        super(SceneMLP, self).__init__()
        
        self.network = nn.Sequential(
            # Flatten input
            nn.Flatten(),
            
            # Hidden Layer 1
            nn.Linear(input_size, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            
            # Hidden Layer 2
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            
            # Hidden Layer 3
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            
            # Output Layer
            nn.Linear(256, n_classes)
        )
    
    def forward(self, x):
        return self.network(x)

# Initialize model
net = SceneMLP(input_size=1*150*150, n_classes=6).to(device)
print(net)
print(f"\nTotal parameters: {sum(p.numel() for p in net.parameters()):,}")

## Step 3: Training Setup

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer: Adam with weight decay for regularization
optimizer = optim.Adam(net.parameters(), lr=0.001, weight_decay=1e-4)

# Learning rate scheduler: reduce LR when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

# Training configuration
num_epochs = 30

## Step 4: Training Loop with Validation

In [ ]:
train_losses = []
val_losses = []
val_accuracies = []
best_val_acc = 0.0

for epoch in range(num_epochs):
    # ---- Training Phase ----
    net.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = net(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
    
    train_loss = running_loss / total_train
    train_acc = correct_train / total_train
    train_losses.append(train_loss)
    
    # ---- Validation Phase ----
    net.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = net(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
    
    val_loss = val_loss / total_val
    val_acc = correct_val / total_val
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    # Step the scheduler
    scheduler.step(val_loss)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(net.state_dict(), 'best_model.pth')
    
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

print(f"\nBest Validation Accuracy: {best_val_acc:.4f}")

### Training and Validation Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses, label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Over Epochs')
ax1.legend()
ax1.grid(True)

ax2.plot(val_accuracies, label='Val Accuracy', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy Over Epochs')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Step 5: Test Set Evaluation (Sanity Check)

In [ ]:
# Load best model
net.load_state_dict(torch.load('best_model.pth'))
net.eval()

correct_test = 0
total_test = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

test_acc = correct_test / total_test
print(f"Test Accuracy: {test_acc:.4f}")

## Step 6: Generate Competition Predictions

In [ ]:
# Ensure best model is loaded
net.load_state_dict(torch.load('best_model.pth'))
net.eval()

preds = []
ids = []

with torch.no_grad():
    for images, names in comp_test_loader:
        images = images.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        preds.extend(predicted.cpu().numpy().tolist())
        ids.extend(list(names))

# Create DataFrame and save CSV
df = pd.DataFrame({'id': ids, 'pred': preds})
csv_file = 'predictions.csv'
df.to_csv(csv_file, index=False)

print(f"CSV file '{csv_file}' created successfully.")
print(f"Total predictions: {len(df)}")
print(f"\nPrediction distribution:")
print(df['pred'].value_counts().sort_index())
print(f"\nFirst 10 rows:")
print(df.head(10))

## Done!
The `predictions.csv` file is saved under `/kaggle/working/` and ready for submission to HuggingFace.